In [3]:
!pip install scikit-learn matplotlib numpy requests pillow

In [8]:
!pip install tensorflow matplotlib numpy pandas scikit-learn

In [10]:
# ✅ Install TensorFlow jika belum ada
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
from tensorflow import keras
import numpy as np

##############################################
# 1️⃣ Membuat Tensor (mirip NumPy)
##############################################
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])
print(t)

##############################################
# 2️⃣ Custom Loss Function → Huber Loss
##############################################
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = 0.5 * tf.square(error)
    linear_loss = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

##############################################
# 3️⃣ Custom Loss Class
##############################################
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = 0.5 * tf.square(error)
        linear_loss = self.threshold * tf.abs(error) - 0.5 * self.threshold ** 2
        return tf.where(is_small_error, squared_loss, linear_loss)

    def get_config(self):
        return {"threshold": self.threshold}

##############################################
# 4️⃣ Custom Activation / Initializer / Regularizer / Constraint
##############################################
def my_softplus(z):
    return tf.math.log(tf.exp(z) + 1.0)

def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2. / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)

def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))

def my_positive_constraint(weights):
    return tf.where(weights < 0., tf.zeros_like(weights), weights)

##############################################
# 5️⃣ Custom Layer Example
##############################################
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight("kernel", shape=[batch_input_shape[-1], self.units], initializer="glorot_normal")
        self.bias = self.add_weight("bias", shape=[self.units], initializer="zeros")
        super().build(batch_input_shape)

    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)

##############################################
# 6️⃣ Custom Model (Example: ResidualBlock)
##############################################
class ResidualBlock(keras.layers.Layer):
    def __init__(self, n_layers, n_neurons, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(n_neurons, activation="elu", kernel_initializer="he_normal") for _ in range(n_layers)]

    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        return inputs + Z

class ResidualRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = keras.layers.Dense(output_dim)

    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(4):
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)

##############################################
# 7️⃣ Gradient Computation Example
##############################################
w1 = tf.Variable(5.)
w2 = tf.Variable(3.)

def f(w1, w2):
    return 3 * w1 ** 2 + 2 * w1 * w2

with tf.GradientTape() as tape:
    z = f(w1, w2)
gradients = tape.gradient(z, [w1, w2])
print("Gradients:", gradients)

##############################################
# 8️⃣ TensorFlow Functions with @tf.function
##############################################
@tf.function
def cube(x):
    return x ** 3

print("Cube(2):", cube(tf.constant(2.)))

tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)
Gradients: [<tf.Tensor: shape=(), dtype=float32, numpy=36.0>, <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]
Cube(2): tf.Tensor(8.0, shape=(), dtype=float32)


# 📚 Chapter 12 - Custom Models and Training with TensorFlow

---

## 🔸 1. TensorFlow Low-Level API
TensorFlow tidak hanya menyediakan Keras API → juga API level rendah untuk membuat **custom models**, **custom loss**, dan **custom training loops**.

---

## 🔸 2. Membuat Tensors
TensorFlow bekerja dengan objek **Tensor** → mirip NumPy array, tapi mendukung **autograd** (automatic differentiation).

t = tf.constant([[1., 2.], [3., 4.]])

---

## 🔸 3. Custom Loss Function
✅ Custom loss function → fleksibilitas untuk membuat loss sendiri.

Contoh:

Huber Loss → kombinasi MSE untuk error kecil, MAE untuk error besar → lebih robust terhadap outlier.  

def huber_fn(y_true, y_pred):
    
---

## 🔸 4. Custom Components
| Customization   | Contoh                     |
| --------------- | -------------------------- |
| **Activation**  | `my_softplus()`            |
| **Initializer** | `my_glorot_initializer()`  |
| **Regularizer** | `my_l1_regularizer()`      |
| **Constraint**  | `my_positive_constraint()` |

---

## 🔸 5. Custom Layer
Dengan mewarisi keras.layers.Layer, kita bisa membuat layer sesuai kebutuhan, misalnya:

class MyDense(keras.layers.Layer):

---

## 🔸 6. Custom Model

✅ Membuat model sendiri → dengan pewarisan keras.Model → contoh implementasi residual block.

class ResidualRegressor(keras.Model):

---

## 🔸 7. Gradient Computation with GradientTape

tf.GradientTape() → untuk menghitung turunan (gradien) secara otomatis.

with tf.GradientTape() as tape:

    z = f(w1, w2)

grads = tape.gradient(z, [w1, w2])

---

## 🔸 8. tf.function for Optimization

Menggunakan @tf.function → konversi Python function menjadi TensorFlow Graph → eksekusi lebih cepat.


✅ Kesimpulan
- Gunakan API low-level → jika butuh fleksibilitas penuh.

- Cocok untuk riset atau eksperimen lanjutan.

- @tf.function → untuk performa → production-grade.
